# Experiment 03 — Encoding Strategies (QNG Optimized)

**Research question:** Which quantum data encoding strategy performs best for particle physics classification when using the Quantum Natural Gradient (QNG)?

We compare three strategies using 2 features / 2 layers / 5000 samples:

| Strategy | Description |
|----------|-------------|
| **Angle encoding** | Ry(xᵢ) — each feature maps to a rotation angle (paper default) |
| **Amplitude encoding** | Features encoded in the amplitudes of the quantum state |
| **Data reuploading** | Features are re-injected at each circuit layer |

Features: col 26 (m_bb) and col 4 (missing energy magnitude).

In [1]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml
from pennylane import numpy as pnp
from sklearn.metrics import roc_auc_score

from utils.data_utils import load_higgs, binary_accuracy

np.random.seed(42)

# Shared hyperparameters
N_FEATURES = 2
N_LAYERS   = 2
N_EPOCHS   = 30
BATCH_SIZE = 32
LR         = 0.05

X_train, X_val, X_test, y_train, y_val, y_test = load_higgs(
    '../data/HIGGS.csv.gz',
    n_features=N_FEATURES,
    feature_indices=[26, 4],
    scale_range=(0, np.pi),
)

Selected features (cols [26, 4]): ['m_bb', 'missing energy mag.']
Dataset: 5000 samples | 2 features | train=3000, val=1000, test=1000


## Training Engine (QNG)

In [2]:
def train_strategy_qng(circuit_fn, n_qubits_for_weights, label):
    """Train a strategy using QNG and return results."""
    pnp.random.seed(42)
    w = pnp.array(
        np.random.uniform(0, 2*np.pi, (N_LAYERS, n_qubits_for_weights, 3)),
        requires_grad=True
    )
    b = pnp.array(0.0, requires_grad=True)
    
    opt_w = qml.QNGOptimizer(stepsize=LR)
    opt_b = qml.AdamOptimizer(stepsize=LR)
    mt_fn = qml.metric_tensor(circuit_fn, approx='block-diag')

    train_losses, val_losses = [], []

    for epoch in range(N_EPOCHS):
        perm = np.random.permutation(len(X_train))
        Xs, ys = X_train[perm], y_train[perm]

        for start in range(0, len(X_train), BATCH_SIZE):
            Xb = Xs[start : start + BATCH_SIZE]
            yb = ys[start : start + BATCH_SIZE].astype(float)

            def cost_w(weights):
                preds = pnp.array([circuit_fn(weights, x) for x in Xb])
                return pnp.mean((yb - (preds + b)) ** 2)
            
            mt = mt_fn(w, Xb[0])
            w = opt_w.step(cost_w, w, metric_tensor_fn=lambda w: mt)

            def cost_b(bias):
                preds = pnp.array([circuit_fn(w, x) for x in Xb])
                return pnp.mean((yb - (preds + bias)) ** 2)
            b = opt_b.step(cost_b, b)

        # Validation Metrics
        val_preds = np.array([float(circuit_fn(w, x) + b) for x in X_val])
        vl_loss = np.mean((y_val.astype(float) - val_preds)**2)
        val_losses.append(vl_loss)

        if (epoch+1) % 10 == 0 or epoch == 0:
            print(f'  [{label}] Epoch {epoch+1}/{N_EPOCHS} | Val Loss: {vl_loss:.4f}')

    # Final Evaluation
    test_raw = np.array([float(circuit_fn(w, x) + b) for x in X_test])
    test_acc = binary_accuracy(y_test, test_raw)
    y_test_01 = (y_test == 1).astype(int)
    test_score = (test_raw - test_raw.min()) / (test_raw.max() - test_raw.min() + 1e-8)
    test_auc = roc_auc_score(y_test_01, test_score)

    print(f'  [{label}] Test Acc: {test_acc:.4f} | Test AUC: {test_auc:.4f}')
    return {'test_acc': test_acc, 'test_auc': test_auc, 'val_losses': val_losses}


## Strategy A — Angle Encoding (paper baseline)

In [3]:
dev_angle = qml.device('default.qubit', wires=N_FEATURES)

@qml.qnode(dev_angle, interface='autograd')
def circuit_angle(weights, x):
    for i in range(N_FEATURES):
        qml.RY(x[i], wires=i)
    for l in range(N_LAYERS):
        for q in range(N_FEATURES):
            qml.Rot(weights[l,q,0], weights[l,q,1], weights[l,q,2], wires=q)
        qml.CNOT(wires=[0, 1])
        qml.CNOT(wires=[1, 0])
    return qml.expval(qml.PauliZ(0))

## Strategy B — Data Reuploading

In [4]:
@qml.qnode(dev_angle, interface='autograd')
def circuit_reupload(weights, x):
    for l in range(N_LAYERS):
        for i in range(N_FEATURES):
            qml.RY(x[i], wires=i)
        for q in range(N_FEATURES):
            qml.Rot(weights[l,q,0], weights[l,q,1], weights[l,q,2], wires=q)
        qml.CNOT(wires=[0, 1])
        qml.CNOT(wires=[1, 0])
    return qml.expval(qml.PauliZ(0))

## Strategy C — Amplitude Encoding

In [5]:
n_amp_qubits = 1
dev_amp = qml.device('default.qubit', wires=n_amp_qubits + 1)

@qml.qnode(dev_amp, interface='autograd')
def circuit_amplitude(weights, x):
    # Note: X must be normalized for AmplitudeEmbedding
    qml.AmplitudeEmbedding(features=x, wires=[0], normalize=True)
    for l in range(N_LAYERS):
        for q in range(n_amp_qubits + 1):
            qml.Rot(weights[l,q,0], weights[l,q,1], weights[l,q,2], wires=q)
        qml.CNOT(wires=[0, 1])
        qml.CNOT(wires=[1, 0])
    return qml.expval(qml.PauliZ(0))

## Run Comparison

In [6]:
print("=== Angle Encoding ===")
res_angle = train_strategy_qng(circuit_angle, N_FEATURES, "Angle")

print("\n=== Data Reuploading ===")
res_reupload = train_strategy_qng(circuit_reupload, N_FEATURES, "Reupload")

print("\n=== Amplitude Encoding ===")
res_amplitude = train_strategy_qng(circuit_amplitude, n_amp_qubits + 1, "Amplitude")

=== Angle Encoding ===
  [Angle] Epoch 1/30 | Val Loss: 0.9752
  [Angle] Epoch 10/30 | Val Loss: 0.9710
  [Angle] Epoch 20/30 | Val Loss: 0.9772
  [Angle] Epoch 30/30 | Val Loss: 0.9699
  [Angle] Test Acc: 0.5990 | Test AUC: 0.6245

=== Data Reuploading ===
  [Reupload] Epoch 1/30 | Val Loss: 0.9710
  [Reupload] Epoch 10/30 | Val Loss: 0.9614
  [Reupload] Epoch 20/30 | Val Loss: 0.9795
  [Reupload] Epoch 30/30 | Val Loss: 0.9859
  [Reupload] Test Acc: 0.5990 | Test AUC: 0.6097

=== Amplitude Encoding ===
  [Amplitude] Epoch 1/30 | Val Loss: 0.9861
  [Amplitude] Epoch 10/30 | Val Loss: 0.9823
  [Amplitude] Epoch 20/30 | Val Loss: 0.9980
  [Amplitude] Epoch 30/30 | Val Loss: 0.9980
  [Amplitude] Test Acc: 0.5530 | Test AUC: 0.5898


In [7]:
print('\nEncoding Strategy Comparison (QNG Optimized)')
print(f'{"Strategy":>15} {"Test Acc":>10} {"Test AUC":>10}')
print('-' * 38)
for name, res in [('Angle', res_angle), ('Reuploading', res_reupload), ('Amplitude', res_amplitude)]:
    print(f'{name:>15} {res["test_acc"]:>10.4f} {res["test_auc"]:>10.4f}')


Encoding Strategy Comparison (QNG Optimized)
       Strategy   Test Acc   Test AUC
--------------------------------------
          Angle     0.5990     0.6245
    Reuploading     0.5990     0.6097
      Amplitude     0.5530     0.5898
